<a href="https://colab.research.google.com/github/emilyperras/econ5200-replication-study/blob/main/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#PHASE2 The Replication Phase: Replicating The DID Engine

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [4]:
import pandas as pd

# Codebook locations for fixed-width file
colspecs = [
(0,3),(4,5),(6,7),(8,9),(10,11),(12,13),(14,15),(16,17),(18,19),(20,21),
(22,24),(25,30),(31,36),(37,42),(43,48),(49,54),(55,60),(61,62),(63,68),(69,70),
(71,76),(77,82),(83,88),(89,94),(95,100),(101,103),(104,106),
(107,108),(109,110),(111,117),(118,120),(121,126),(127,132),(133,138),(139,144),
(145,150),(151,156),(157,158),(159,160),(161,166),(167,172),(173,178),(179,184),
(185,190),(191,193),(194,196)
]

names = [
"sheet","chain","co_owned","state","southj","centralj","northj","pa1","pa2","shore",
"ncalls","empft","emppt","nmgrs","wage_st","inctime","firstinc","bonus","pctaff","meals",
"open","hrsopen","psoda","pfry","pentree","nregs","nregs11",
"type2","status2","date2","ncalls2","empft2","emppt2","nmgrs2","wage_st2",
"inctime2","firstin2","special2","meals2","open2r","hrsopen2","psoda2","pfry2",
"pentree2","nregs2","nregs112"
]

df = pd.read_fwf("public.dat", colspecs=colspecs, names=names)

df.head()

,sheet,chain,co_owned,state,southj,centralj,northj,pa1,pa2,shore,...,firstin2,special2,meals2,open2r,hrsopen2,psoda2,pfry2,pentree2,nregs2,nregs112
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1,2,6.50,16.50,1.03,.,0.94,4,4
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0,2,10.00,13.00,1.01,0.89,2.35,4,4
2,506,2,1,0,0,0,0,1,0,0,...,0.25,.,1,11.00,11.00,0.95,0.74,2.33,4,3
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,0.92,0.79,0.87,2,2
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,1.01,0.84,0.95,2,2


Replace "." values to NaN

In [5]:
df["empft"] = pd.to_numeric(df["empft"], errors="coerce")
df["emppt"] = pd.to_numeric(df["emppt"], errors="coerce")
df["nmgrs"] = pd.to_numeric(df["nmgrs"], errors="coerce")

df["empft2"] = pd.to_numeric(df["empft2"], errors="coerce")
df["emppt2"] = pd.to_numeric(df["emppt2"], errors="coerce")
df["nmgrs2"] = pd.to_numeric(df["nmgrs2"], errors="coerce")

##Step 1: Descriptive Table Reconstruction (Table 2)

Full-Time Equivalent employment (FTE) for each restaurant before and after the minimum wage increase: FTE = full-time workers + managers + 0.5 × part-time workers

In [6]:
df["fte_before"] = df["empft"] + df["nmgrs"] + 0.5 * df["emppt"]
df["fte_after"] = df["empft2"] + df["nmgrs2"] + 0.5 * df["emppt2"]

df[["sheet","fte_before","fte_after"]].head()

,sheet,fte_before,fte_after
0,46,40.50,24.0
1,49,13.75,11.5
2,506,8.50,10.5
3,56,34.00,20.0
4,61,24.00,35.5


##Step 2: The Simple Difference

Calculate average employment before and after for each state

In [7]:
nj_before = df[df["state"] == 1]["fte_before"].mean()
nj_after = df[df["state"] == 1]["fte_after"].mean()

pa_before = df[df["state"] == 0]["fte_before"].mean()
pa_after = df[df["state"] == 0]["fte_after"].mean()

print("NJ before:", nj_before)
print("NJ after:", nj_after)
print("PA before:", pa_before)
print("PA after:", pa_after)

NJ before: 20.439408099688475
NJ after: 21.02742946708464
PA before: 23.33116883116883
PA after: 21.165584415584416


##Step 3: Regression Implementation

In [8]:
did = (nj_after - nj_before) - (pa_after - pa_before)

print("Manual Difference-in-Differences:", did)

Manual Difference-in-Differences: 2.753605782980582


In [9]:
# Create copies and add variable to mark before and after treatment
pre = df[["sheet","state","fte_before"]].copy()
pre["t"] = 0

post = df[["sheet","state","fte_after"]].copy()
post["t"] = 1

In [10]:
# Match Collum Names
pre.columns = ["store_id","treat","fte","t"]
post.columns = ["store_id","treat","fte","t"]

In [11]:
# Combine before and after data
data = pd.concat([pre, post])

Run Regression

In [12]:
# Outcome ~ treatment + time + treatment × time
model = smf.ols("fte ~ treat * t", data=data)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    fte   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.964
Date:                Wed, 11 Mar 2026   Prob (F-statistic):              0.118
Time:                        02:49:15   Log-Likelihood:                -2904.2
No. Observations:                 794   AIC:                             5816.
Df Residuals:                     790   BIC:                             5835.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.072     21.767      0.0

Step 4: Standard Errors and Clustering

In [13]:
# Drop missing values
reg_data = data[["store_id", "fte", "treat", "t"]].dropna()

model = smf.ols("fte ~ treat * t", data=reg_data)

# Cluster by store using statsmodels
cluster_results = model.fit(cov_type="cluster", cov_kwds={"groups": reg_data["store_id"]})

print(cluster_results.summary())

                            OLS Regression Results                            
Dep. Variable:                    fte   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.806
Date:                Wed, 11 Mar 2026   Prob (F-statistic):              0.146
Time:                        02:49:15   Log-Likelihood:                -2904.2
No. Observations:                 794   AIC:                             5816.
Df Residuals:                     790   BIC:                             5835.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.347     17.327      0.0